# Data Cleaning - Missing Values & Duplicates

## Key Concepts
### 1. Missing Data Types
### 2. Handling Strategies
### 3. Duplicates

### Step 0: Reload the data
- Start a new cell and load the data again.

In [45]:
import pandas as pd
import numpy as np

In [46]:
movies = pd.read_csv("movies-new.csv")

# Preview the missing summary
print(movies.isnull().sum())

Rank                    0
Title                   0
Studio                  0
Gross                   0
Year                    0
Budget                  1
Runtime                 1
Genre                   0
Director                0
Actors                  0
ReleaseDate             0
Country                 0
Language                0
Rating                  1
Votes                   0
Metascore               0
MPAA                    1
Awards                 80
Sequel                  0
Franchise              21
Profit                  0
ROI                     0
ReleaseMonth            0
ReleaseDayOfWeek        0
Age                     0
ProductionCountries     0
WeekendGross            0
dtype: int64


**Mission**: Clean these missing values systematically.

### Step 1: Understand the context of each missing value

Ask: *Why is the value missing?*

- **Budget** - Data entry error or not publicly disclosed.
- **Runtime** - Same row as *Budet*? Check
- **Rating** - IMDb rating not avaialble for some movies (maybe new releases or not rated).
- **Franchise** - The movie does NOT belong to a franchise - this is **not a missing value**, its *"not applicable"*.
- **Profit** - Profit = Gross - Budget. If Budget is missing, Profit cannot be calculated.
- **ROI** - ROI = Profit / Budget. If Budget is missing, ROI cannot be calculated.

#### Action plan:
- *Franchise*: Fill with "No Franchise" (constant fill).
- *Budget*: Impute with **median** (less affected by outliers).
- *Runtime*: Impute with **median**.
- *Rating*: Impute with **mean** (or median - we'll use median)
- *Profit*: Recalucate after Budget is filled.
- *ROI*: Recalculate after Profit and Buget are filled.

### Step 2: Inspect the row with missing Budget and Runtime

In [47]:
movies["Budget"]

0     356000000.0
1     237000000.0
2     200000000.0
3     245000000.0
4     300000000.0
         ...     
95    230000000.0
96    125000000.0
97    165000000.0
98    200000000.0
99    145000000.0
Name: Budget, Length: 100, dtype: float64

In [48]:
movies["Budget"].isna()

0     False
1     False
2     False
3     False
4     False
      ...  
95    False
96    False
97    False
98    False
99    False
Name: Budget, Length: 100, dtype: bool

In [49]:
movies[movies["Budget"].isna()]

,Rank,Title,Studio,Gross,Year,Budget,Runtime,Genre,Director,Actors,...,Awards,Sequel,Franchise,Profit,ROI,ReleaseMonth,ReleaseDayOfWeek,Age,ProductionCountries,WeekendGross
5,6,Jurassic World,Universal,"$1,671.70",2015,NaN,124.0,"Action, Adventure, Sci-Fi",Colin Trevorrow,"Chris Pratt, Bryce Dallas Howard",...,NaN,1,Jurassic Park,1521700000,10.14,6,Friday,11,USA,140000000


In [50]:
missing_budget = movies[movies["Budget"].isna()]
missing_budget

,Rank,Title,Studio,Gross,Year,Budget,Runtime,Genre,Director,Actors,...,Awards,Sequel,Franchise,Profit,ROI,ReleaseMonth,ReleaseDayOfWeek,Age,ProductionCountries,WeekendGross
5,6,Jurassic World,Universal,"$1,671.70",2015,NaN,124.0,"Action, Adventure, Sci-Fi",Colin Trevorrow,"Chris Pratt, Bryce Dallas Howard",...,NaN,1,Jurassic Park,1521700000,10.14,6,Friday,11,USA,140000000


In [51]:
missing_budget[["Rank", "Title", "Studio", "Year", "Budget", "Runtime"]]

,Rank,Title,Studio,Year,Budget,Runtime
5,6,Jurassic World,Universal,2015,NaN,124.0


In [52]:
movies["Runtime"].isna()

0     False
1     False
2     False
3     False
4     False
      ...  
95    False
96    False
97    False
98    False
99    False
Name: Runtime, Length: 100, dtype: bool

In [53]:
missing_runtime = movies[movies["Runtime"].isna()]
missing_runtime[["Rank", "Title", "Runtime"]]

,Rank,Title,Runtime
9,10,Black Panther,NaN


In [54]:
# Check profit and ROI missing rows
missing_profit = movies[movies["Profit"].isna()]
missing_profit

,Rank,Title,Studio,Gross,Year,Budget,Runtime,Genre,Director,Actors,...,Awards,Sequel,Franchise,Profit,ROI,ReleaseMonth,ReleaseDayOfWeek,Age,ProductionCountries,WeekendGross


In [55]:
missing_ROI = movies[movies["ROI"].isna()]
missing_ROI

,Rank,Title,Studio,Gross,Year,Budget,Runtime,Genre,Director,Actors,...,Awards,Sequel,Franchise,Profit,ROI,ReleaseMonth,ReleaseDayOfWeek,Age,ProductionCountries,WeekendGross


In [56]:
missing_budget

,Rank,Title,Studio,Gross,Year,Budget,Runtime,Genre,Director,Actors,...,Awards,Sequel,Franchise,Profit,ROI,ReleaseMonth,ReleaseDayOfWeek,Age,ProductionCountries,WeekendGross
5,6,Jurassic World,Universal,"$1,671.70",2015,NaN,124.0,"Action, Adventure, Sci-Fi",Colin Trevorrow,"Chris Pratt, Bryce Dallas Howard",...,NaN,1,Jurassic Park,1521700000,10.14,6,Friday,11,USA,140000000


### Step 3: Fill `Franchise` - constant fill

In [57]:
movies

,Rank,Title,Studio,Gross,Year,Budget,Runtime,Genre,Director,Actors,...,Awards,Sequel,Franchise,Profit,ROI,ReleaseMonth,ReleaseDayOfWeek,Age,ProductionCountries,WeekendGross
0,1,Avengers: Endgame,Buena Vista,"$2,796.30",2019,356000000.0,181.0,"Action, Adventure, Sci-Fi",Anthony Russo,"Robert Downey Jr., Chris Evans, Mark Ruffalo",...,Oscar for Best Visual Effects,1,Marvel,2440300000,6.85,4,Friday,7,USA,150000000
1,2,Avatar,Fox,"$2,789.70",2009,237000000.0,162.0,"Action, Adventure, Fantasy",James Cameron,"Sam Worthington, Zoe Saldana, Sigourney Weaver",...,Oscar for Best Cinematography,0,NaN,2522700000,10.64,12,Friday,17,USA,180000000
2,3,Titanic,Paramount,"$2,187.50",1997,200000000.0,195.0,"Drama, Romance",James Cameron,"Leonardo DiCaprio, Kate Winslet, Billy Zane",...,Oscar for Best Picture,0,NaN,1987500000,9.94,12,Friday,29,USA,120000000
3,4,Star Wars: The Force Awakens,Buena Vista,"$2,068.20",2015,245000000.0,138.0,"Action, Adventure, Sci-Fi",J.J. Abrams,"Daisy Ridley, John Boyega, Oscar Isaac",...,NaN,1,Star Wars,1823200000,7.44,12,Friday,11,USA,170000000
4,5,Avengers: Infinity War,BV,"$2,048.40",2018,300000000.0,149.0,"Action, Adventure, Sci-Fi",Anthony Russo,"Robert Downey Jr., Chris Hemsworth, Josh Brolin",...,NaN,1,Marvel,1748400000,5.83,4,Friday,8,USA,190000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,The Amazing Spider-Man,Sony,$757.90,2012,230000000.0,136.0,"Action, Adventure, Sci-Fi",Marc Webb,"Andrew Garfield, Emma Stone, Rhys Ifans",...,NaN,1,Spider-Man,527900000,2.30,7,Tuesday,14,USA,50000000
96,97,The Hunger Games: Mockingjay - Part 1,Lionsgate,$755.40,2014,125000000.0,123.0,"Adventure, Sci-Fi, Thriller",Francis Lawrence,"Jennifer Lawrence, Josh Hutcherson, Liam Hemsw...",...,NaN,1,Hunger Games,630400000,5.04,11,Friday,12,USA,50000000
97,98,Shrek Forever After,Dreamworks,$752.60,2010,165000000.0,93.0,"Animation, Adventure, Comedy",Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz",...,NaN,1,Shrek,587600000,3.56,5,Friday,16,USA,40000000
98,99,X-Men: Days of Future Past,Fox,$747.90,2014,200000000.0,132.0,"Action, Adventure, Sci-Fi",Bryan Singer,"Hugh Jackman, James McAvoy, Michael Fassbender",...,NaN,1,X-Men,547900000,2.74,5,Friday,12,USA,60000000


In [58]:
movies.isna().sum()

Rank                    0
Title                   0
Studio                  0
Gross                   0
Year                    0
Budget                  1
Runtime                 1
Genre                   0
Director                0
Actors                  0
ReleaseDate             0
Country                 0
Language                0
Rating                  1
Votes                   0
Metascore               0
MPAA                    1
Awards                 80
Sequel                  0
Franchise              21
Profit                  0
ROI                     0
ReleaseMonth            0
ReleaseDayOfWeek        0
Age                     0
ProductionCountries     0
WeekendGross            0
dtype: int64

- Since `Franchise` has 21 missing values and they represent "not a franchise", we fill with a new category:

In [59]:
# Fill missing Franchise with 'No Frachise'
# Using fillna(value) method; fillna stands for fill the missing values
movies["Franchise"] = movies["Franchise"].fillna("No Franchise")

In [60]:
movies.head(7)

,Rank,Title,Studio,Gross,Year,Budget,Runtime,Genre,Director,Actors,...,Awards,Sequel,Franchise,Profit,ROI,ReleaseMonth,ReleaseDayOfWeek,Age,ProductionCountries,WeekendGross
0,1,Avengers: Endgame,Buena Vista,"$2,796.30",2019,356000000.0,181.0,"Action, Adventure, Sci-Fi",Anthony Russo,"Robert Downey Jr., Chris Evans, Mark Ruffalo",...,Oscar for Best Visual Effects,1,Marvel,2440300000,6.85,4,Friday,7,USA,150000000
1,2,Avatar,Fox,"$2,789.70",2009,237000000.0,162.0,"Action, Adventure, Fantasy",James Cameron,"Sam Worthington, Zoe Saldana, Sigourney Weaver",...,Oscar for Best Cinematography,0,No Franchise,2522700000,10.64,12,Friday,17,USA,180000000
2,3,Titanic,Paramount,"$2,187.50",1997,200000000.0,195.0,"Drama, Romance",James Cameron,"Leonardo DiCaprio, Kate Winslet, Billy Zane",...,Oscar for Best Picture,0,No Franchise,1987500000,9.94,12,Friday,29,USA,120000000
3,4,Star Wars: The Force Awakens,Buena Vista,"$2,068.20",2015,245000000.0,138.0,"Action, Adventure, Sci-Fi",J.J. Abrams,"Daisy Ridley, John Boyega, Oscar Isaac",...,NaN,1,Star Wars,1823200000,7.44,12,Friday,11,USA,170000000
4,5,Avengers: Infinity War,BV,"$2,048.40",2018,300000000.0,149.0,"Action, Adventure, Sci-Fi",Anthony Russo,"Robert Downey Jr., Chris Hemsworth, Josh Brolin",...,NaN,1,Marvel,1748400000,5.83,4,Friday,8,USA,190000000
5,6,Jurassic World,Universal,"$1,671.70",2015,NaN,124.0,"Action, Adventure, Sci-Fi",Colin Trevorrow,"Chris Pratt, Bryce Dallas Howard",...,NaN,1,Jurassic Park,1521700000,10.14,6,Friday,11,USA,140000000
6,7,Marvel's The Avengers,Buena Vista,"$1,518.80",2012,220000000.0,143.0,"Action, Adventure, Sci-Fi",Joss Whedon,"Robert Downey Jr., Chris Evans, Scarlett Johan...",...,NaN,1,Marvel,1298800000,5.90,5,Friday,14,USA,160000000


In [61]:
movies.isna().sum()

Rank                    0
Title                   0
Studio                  0
Gross                   0
Year                    0
Budget                  1
Runtime                 1
Genre                   0
Director                0
Actors                  0
ReleaseDate             0
Country                 0
Language                0
Rating                  1
Votes                   0
Metascore               0
MPAA                    1
Awards                 80
Sequel                  0
Franchise               0
Profit                  0
ROI                     0
ReleaseMonth            0
ReleaseDayOfWeek        0
Age                     0
ProductionCountries     0
WeekendGross            0
dtype: int64

In [62]:
movies["Franchise"].isna().sum()

0

In [63]:
movies["Franchise"].value_counts()

Franchise
No Franchise                21
Marvel                      11
Harry Potter                 8
Star Wars                    6
Spider-Man                   6
Pirates of the Caribbean     4
Shrek                        3
The Hobbit                   3
Jurassic Park                3
Despicable Me                3
Fast & Furious               3
Lord of the Rings            3
Transformers                 3
DC                           3
James Bond                   2
Batman                       2
Deadpool                     2
Ice Age                      2
Hunger Games                 2
X-Men                        1
Fantastic Beasts             1
Indiana Jones                1
Mission: Impossible          1
Incredibles                  1
Twilight                     1
Wolf Warrior                 1
Finding Nemo                 1
Toy Story                    1
Madagascar                   1
Name: count, dtype: int64

### Step 4: Fill `Budget` - mean imputation

- First, check the distribution of `Budget` to decide between mean (average) and median

In [64]:
# mean
movies["Budget"].mean()

160752525.25252524

In [65]:
# median
movies["Budget"].median()

170000000.0

- The mean is higher than meidan, indicating right-skewed (a few huge budgets)
- The mean is lesser than median, indicating left-skweed

In [66]:
# mean is lesser, so we use mean value to fill the missing value
budget_mean = movies["Budget"].mean()
movies["Budget"] = movies["Budget"].fillna(budget_mean)

In [67]:
movies["Budget"].isna().sum()

0

In [68]:
missing_runtime

,Rank,Title,Studio,Gross,Year,Budget,Runtime,Genre,Director,Actors,...,Awards,Sequel,Franchise,Profit,ROI,ReleaseMonth,ReleaseDayOfWeek,Age,ProductionCountries,WeekendGross
9,10,Black Panther,Buena Vista,"$1,346.90",2018,200000000.0,NaN,"Action, Adventure, Sci-Fi",Ryan Coogler,"Chadwick Boseman, Michael B. Jordan, Lupita Ny...",...,Oscar for Best Costume Design,1,Marvel,1146900000,5.73,2,Friday,8,USA,110000000


### Step 5: Fill `Runtime` - mean imputation

In [69]:
print("Runtime mean:", movies["Runtime"].mean())
print("Runtime median:", movies["Runtime"].median())

Runtime mean: 132.04040404040404
Runtime median: 133.0


In [70]:
runtime_mean = movies["Runtime"].mean()
movies["Runtime"] = movies["Runtime"].fillna(runtime_mean)

In [71]:
movies["Runtime"].isnull().sum()

0

### Step 6: Fill `Rating` - mean imputation

In [72]:
print("Rating mean:", movies["Rating"].mean())
print("Rating median:", movies["Rating"].median())

Rating mean: 7.3747474747474735
Rating median: 7.4


- The median (7.4) is a clean value, so use median.

In [73]:
rating_median = movies["Rating"].median()
movies["Rating"] = movies["Rating"].fillna(rating_median)

In [74]:
movies["Rating"].isnull().sum()

0

### Step 7: Fix `Profit` and `ROI` - recalculate after filling Budget

- Actually, a better approach: Since we have `Gross` as string, `Profit` was computed incorrectly by the data provider.
- We will drop the `Profit` and `ROI` columns completely, and recreate them.

In [75]:
# Drop Profit and ROI columns (we'll recompute them after cleaning Gross)
movies = movies.drop(columns=["Profit", "ROI"])
movies.shape

(100, 25)

In [77]:
print(movies.columns.tolist())

['Rank', 'Title', 'Studio', 'Gross', 'Year', 'Budget', 'Runtime', 'Genre', 'Director', 'Actors', 'ReleaseDate', 'Country', 'Language', 'Rating', 'Votes', 'Metascore', 'MPAA', 'Awards', 'Sequel', 'Franchise', 'ReleaseMonth', 'ReleaseDayOfWeek', 'Age', 'ProductionCountries', 'WeekendGross']


### Step 8: Check for duplicates

- We have 100 rows. Are any rows exact duplicates?

In [82]:
movies.duplicated()

0     False
1     False
2     False
3     False
4     False
      ...  
95    False
96    False
97    False
98    False
99    False
Length: 100, dtype: bool

In [88]:
# Check for duplicate rows
duplicates = movies.duplicated()
print("Number of duplicate rows:", duplicates.sum())

Number of duplicate rows: 0


- No duplicates - but let's check for duplicates in `Title` (since a movie shouldn't appear twice):

In [91]:
duplicate_titles = movies["Title"].duplicated()
print("Duplicated titles count:", duplicate_titles.sum())

Duplicated titles count: 0


In [93]:
movies[duplicate_titles][["Rank", "Title"]]

,Rank,Title


### Step 9: (Optional) Check for duplicates in a subset

- Sometiesm you might want to check duplicates based on a subset of columns, e.g., ['Title', 'Year']:

In [94]:
subset_dups = movies.duplicated(subset=['Title', 'Year'])
print("Duplicates based on Title+year:", subset_dups.sum())

Duplicates based on Title+year: 0


- So the dataset is clean of duplicates

### Step 10: The complete cleaning summary

- Let's re-run `.info()` to see our progress:

In [95]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Rank                 100 non-null    int64  
 1   Title                100 non-null    object 
 2   Studio               100 non-null    object 
 3   Gross                100 non-null    object 
 4   Year                 100 non-null    int64  
 5   Budget               100 non-null    float64
 6   Runtime              100 non-null    float64
 7   Genre                100 non-null    object 
 8   Director             100 non-null    object 
 9   Actors               100 non-null    object 
 10  ReleaseDate          100 non-null    object 
 11  Country              100 non-null    object 
 12  Language             100 non-null    object 
 13  Rating               100 non-null    float64
 14  Votes                100 non-null    int64  
 15  Metascore            100 non-null    int6

- Gross is still string - we will fix that later

### Step 11: Save the cleaned dataset (without the dropped columns)

In [96]:
movies.to_csv("movies_cleaned.csv", index=False)
print("Cleaned dataset saved as 'movies-cleaned.csv'")

Cleaned dataset saved as 'movies-cleaned.csv'


### Summary

- **Missing values** come in different forms - some are truly missing, others not "not applicable".
- **Constant filling** (for `Franchise`).
- **Median imputation** (for `Budget`, `Runtime`, `Rating`)
- **Dropping columns** that are unreliable or will be recalculated (`Profit`, `ROI`).
- **Deleting duplicates** with `.duplicated()` and removing them with `.drop_duplicates()` (though we had none).
- **Saving** a cleaned dataset for the next level.

- **Golden rule**: Always understand the reason for missingness before choosing a strategy. Never impute blindly - and document your decisions.